In [ ]:
# ================================================================================
# KAGGLE CONFIG (FULL ARGS MIRROR OF src/2d/train.py)
# ================================================================================
from pathlib import Path

# Kaggle I/O only
DATA_DIR = "/kaggle/input/datasets/hngphm007/2d-mri-1/processed_oasis_2d_jpeg"
WORKING_DIR = Path("/kaggle/working")

# Notebook extra (model selector)
MODEL_CHOICE = "resnet"  # "densenet", "resnet", "mobilenet"
NUM_FOLDS = 5

# Full argument set aligned with src/2d/train.py
args = {
    # Data
    "data_dir": DATA_DIR,
    "fold": -1,
    "view": "all",
    "num_slices_axial": 80,
    "num_slices_coronal": 80,
    "num_slices_sagittal": 80,

    # Model
    "embed_dim": 256,
    "freeze_backbone": True,
    "trainable_layers": [],

    # Training
    "epochs": 150,
    "batch_size": 4,
    "lr": 2e-5,
    "weight_decay": 1e-2,
    "optimizer": "adamw",
    "scheduler": "plateau",
    "step_size": 40,
    "gamma": 0.1,
    "use_class_weights": False,
    "label_smoothing": 0.0,
    "use_amp": False,

    # Directories
    "checkpoint_dir": str(WORKING_DIR / "checkpoints" / "2d_cv5_notebook"),
    "log_dir": str(WORKING_DIR / "logs" / "2d_cv5_notebook"),

    # Other
    "num_workers": 2,
    "seed": 0,
    "save_freq": 10,
    "early_stopping_patience": 50,
    "resume": False,

    # Wandb
    "use_wandb": True,
    "wandb_project": "alzheimer-2d-classification",
    "exp_name": "2d_cnn_attention_notebook",
}

Path(args["checkpoint_dir"]).mkdir(parents=True, exist_ok=True)
Path(args["log_dir"]).mkdir(parents=True, exist_ok=True)

print("Kaggle DATA_DIR:", args["data_dir"] )
print("Checkpoint dir:", args["checkpoint_dir"] )
print("Log dir:", args["log_dir"] )
print("MODEL_CHOICE:", MODEL_CHOICE)
print("Full args:")

for k, v in args.items():    print(f"  {k}: {v}")

In [ ]:
# ================================================================================
# IMPORTS
# ================================================================================
import os
import json
import random
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import Compose, RandomAffine
from tqdm import tqdm
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, auc
)
from PIL import Image

# Optional dependencies
try:
    from torch.utils.tensorboard import SummaryWriter
    HAS_TENSORBOARD = True
except ImportError:
    HAS_TENSORBOARD = False
    SummaryWriter = None

try:
    import wandb
    HAS_WANDB = True
except ImportError:
    HAS_WANDB = False
    wandb = None

print(f"[INFO] HAS_TENSORBOARD={HAS_TENSORBOARD}, HAS_WANDB={HAS_WANDB}")

In [ ]:
# ================================================================================
# DATASET CLASS (FROM src/2d/dataset.py)
# ================================================================================

class _ImageAccessor:
    def __init__(self, dataset):
        self._ds = dataset

    def __getitem__(self, idx):
        subject_id = self._ds.subject_ids[idx]
        slices = []
        for v in self._ds.load_views:
            for fpath in self._ds.file_index[subject_id][v]:
                slices.append(np.array(Image.open(fpath)))
        return np.stack(slices, axis=0)


class OASIS2DDataset(Dataset):
    VIEWS = ["axial", "coronal", "sagittal"]
    CLASSES = {"normal": 0, "nonnormal": 1}

    def __init__(self, data_dir, fold=0, split="train", view="all", one_transform=None, all_transform=None):
        self.data_dir = Path(data_dir)
        self.fold = fold
        self.split = split
        self.view = view
        self.one_transform = one_transform
        self.all_transform = all_transform

        self.load_views = self.VIEWS if view == "all" else [view]

        primary_view = self.load_views[0]
        subject_label_map = {}

        for cls_name, label in self.CLASSES.items():
            cls_dir = self.data_dir / primary_view / f"fold_{fold}" / split / cls_name
            if not cls_dir.exists():
                raise FileNotFoundError(f"Directory not found: {cls_dir}")
            for fpath in sorted(cls_dir.glob("*.jpg")):
                parts = fpath.stem.split("_")
                subject_id = "_".join(parts[:3])
                subject_label_map[subject_id] = label

        self.subject_ids = sorted(subject_label_map.keys())
        self.labels = np.array([subject_label_map[s] for s in self.subject_ids], dtype=np.int64)

        self.file_index = {}
        for subject_id in self.subject_ids:
            label = subject_label_map[subject_id]
            cls_name = "normal" if label == 0 else "nonnormal"
            self.file_index[subject_id] = {}
            for v in self.load_views:
                view_dir = self.data_dir / v / f"fold_{fold}" / split / cls_name
                files = sorted(view_dir.glob(f"{subject_id}_{v}_*.jpg"))
                if not files:
                    raise FileNotFoundError(
                        f"No JPEG files found for subject '{subject_id}' view '{v}' in {view_dir}"
                    )
                self.file_index[subject_id][v] = files

        self.images = _ImageAccessor(self)
        first_subj = self.subject_ids[0]
        self._n_slices_per_view = len(self.file_index[first_subj][self.load_views[0]])
        self._n_slices_total = self._n_slices_per_view * len(self.load_views)

        print(f"\n--- Loaded {split} Fold {fold} (view={view}) ---")
        print(f"Subjects: {len(self.subject_ids)}")
        print(f"Slices/subject: {self._n_slices_total}")
        print(f"Normal: {np.sum(self.labels == 0)}")
        print(f"Alzheimer: {np.sum(self.labels == 1)}")

    def __len__(self):
        return len(self.subject_ids)

    def __getitem__(self, idx):
        subject_id = self.subject_ids[idx]
        label = int(self.labels[idx])
        slices = []

        for v in self.load_views:
            for fpath in self.file_index[subject_id][v]:
                pil_img = Image.open(fpath)
                if self.one_transform is not None:
                    pil_img = self.one_transform(pil_img)

                if isinstance(pil_img, Image.Image):
                    arr = np.array(pil_img, dtype=np.float32) / 255.0
                    slice_tensor = torch.from_numpy(arr).unsqueeze(0)
                elif torch.is_tensor(pil_img):
                    slice_tensor = pil_img.float()
                    if slice_tensor.ndim == 2:
                        slice_tensor = slice_tensor.unsqueeze(0)
                else:
                    raise TypeError(f"Unsupported type after one_transform: {type(pil_img)}")

                slices.append(slice_tensor)

        image = torch.stack(slices, dim=0)
        if self.all_transform is not None:
            image = self.all_transform(image)

        return image, torch.tensor(label, dtype=torch.long), subject_id

print("[INFO] Dataset class ready")

In [ ]:
# ================================================================================
# MODELS (FROM src/2d/models.py)
# ================================================================================
from torchvision.models import (
    densenet121, DenseNet121_Weights,
    resnet18,
    mobilenet_v3_small,
    resnet50, ResNet50_Weights
  )


class MyDenseNetMultiAttention(nn.Module):
    def __init__(self, num_classes=2, num_slices=80, embed_dim=256):
        super().__init__()
        base_model = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1)
        original_conv = base_model.features.conv0
        base_model.features.conv0 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        base_model.features.conv0.weight.data = original_conv.weight.data.sum(dim=1, keepdim=True)

        self.backbone = base_model.features
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.proj = nn.Linear(1024, embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=8, batch_first=True)
        self.classifier = nn.Sequential(nn.LayerNorm(embed_dim), nn.Linear(embed_dim, num_classes))

    def forward(self, x):
        B, S, C, H, W = x.shape
        x = x.view(B * S, C, H, W)
        features = self.backbone(x)
        features = self.avgpool(features).flatten(1)
        features = features.view(B, S, 1024)
        features = self.proj(features)
        attn_out, attn_weights = self.attn(features, features, features)
        out = (features + attn_out).mean(dim=1)
        logits = self.classifier(out)
        return logits, attn_weights


class MyResNetMultiAttention(nn.Module):
    """
    ResNet-based slice encoder with self-attention aggregation for
    3D MRI classification.

    Pipeline:
        1. Slice-wise feature extraction using pretrained ResNet50
        2. Slice embedding projection
        3. Multi-head self-attention across slices
        4. Global slice pooling
        5. Classification head

    Expected input shape:
        (B, S, 1, H, W)

        B = batch size
        S = number of slices per volume
    """

    def __init__(self, num_classes=2, num_slices=80, embed_dim=256):
        super().__init__()

        # ------------------------------------------------------------------
        # 1. CNN Backbone (ResNet50 pretrained on ImageNet)
        # ------------------------------------------------------------------
        # base_model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        base_model = resnet18(weights=None)

        self.feature_dim = base_model.fc.in_features
        

        # Convert first convolution layer to accept grayscale input (1 channel)
        original_conv = base_model.conv1
        base_model.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=64,
            kernel_size=7,
            stride=2,
            padding=3,
            bias=False,
        )

        # Initialize grayscale weights by summing RGB pretrained weights
        base_model.conv1.weight.data = original_conv.weight.data.sum(
            dim=1, keepdim=True
        )

        # Remove the final FC layer; keep everything up to the global avg pool
        self.backbone = nn.Sequential(
            base_model.conv1,
            base_model.bn1,
            base_model.relu,
            base_model.maxpool,
            base_model.layer1,
            base_model.layer2,
            base_model.layer3,
            base_model.layer4,
        )
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        # ------------------------------------------------------------------
        # 2. Slice-level Attention Module
        # ------------------------------------------------------------------
        self.proj = nn.Linear(self.feature_dim, embed_dim)

        self.pos_embed = nn.Parameter(torch.zeros(1, num_slices, embed_dim))
        self.attn = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=8,
            batch_first=True,
        )

        # ------------------------------------------------------------------
        # 3. Classification Head
        # ------------------------------------------------------------------
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, num_classes),
        )

    def forward(self, x):
        """
        Forward pass.

        Args:
            x: MRI volume tensor
               Shape = (B, S, 1, H, W)

        Returns:
            logits: classification output (B, num_classes)
            attn_weights: attention map across slices
                          Shape = (B, num_heads, S, S)
        """

        B, S, C, H, W = x.shape

        # --------------------------------------------------------------
        # Slice-wise CNN feature extraction
        # --------------------------------------------------------------
        x = x.view(B * S, C, H, W)

        features = self.backbone(x)          # (B*S, 2048, H', W')
        features = self.avgpool(features)    # (B*S, 2048, 1, 1)
        features = features.flatten(1)       # (B*S, 2048)

        # Restore slice sequence structure
        features = features.view(B, S, self.feature_dim)

        # --------------------------------------------------------------
        # Slice embedding projection
        # --------------------------------------------------------------
        features = self.proj(features)       # (B, S, embed_dim)

        feature = features + self.pos_embed

        # --------------------------------------------------------------
        # Multi-head self-attention across slices
        # --------------------------------------------------------------
        attn_out, attn_weights = self.attn(
            features,
            features,
            features,
        )

        # Residual connection + global slice pooling
        out = (features + attn_out).mean(dim=1)   # (B, embed_dim)

        # --------------------------------------------------------------
        # Classification
        # --------------------------------------------------------------
        logits = self.classifier(out)

        return logits, attn_weights


class MyMobileNetMultiAttention(nn.Module):
    def __init__(self, num_classes=2, num_slices=80, embed_dim=256):
        super().__init__()
        base_model = mobilenet_v3_small(weights=None)
        base_model.features[0][0] = nn.Conv2d(1, 16, 3, stride=2, padding=1, bias=False)

        self.backbone = base_model.features
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.feature_dim = base_model.classifier[0].in_features
        self.proj = nn.Linear(self.feature_dim, embed_dim)
        self.pos_embed = nn.Parameter(torch.zeros(1, num_slices, embed_dim))
        self.attn = nn.MultiheadAttention(embed_dim, num_heads=4, batch_first=True)
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(p=0.5),
            nn.Linear(embed_dim, num_classes),
        )

    def forward(self, x):
        B, S, C, H, W = x.shape
        x = x.view(B * S, C, H, W)
        features = self.backbone(x)
        features = self.avgpool(features).flatten(1)
        features = features.view(B, S, -1)
        embeddings = self.proj(features) + self.pos_embed
        attn_out, attn_weights = self.attn(embeddings, embeddings, embeddings)
        out = (embeddings + attn_out).mean(dim=1)
        logits = self.classifier(out)
        return logits, attn_weights

print("[INFO] Model classes ready")

In [ ]:
# ================================================================================
# TRAIN/VAL UTILS + PLOTS
# ================================================================================

def save_checkpoint(model, optimizer, epoch, metric, path):
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "metric": metric,
    }, path)


def load_checkpoint(path, model, optimizer=None, device="cpu"):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"] )
    if optimizer is not None and "optimizer_state_dict" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"] )
    return ckpt.get("epoch", 0), ckpt.get("metric", 0.0)


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)


def get_class_weights(labels):
    classes, counts = np.unique(labels, return_counts=True)
    total = counts.sum()
    weights = total / (len(classes) * counts)
    return weights.astype(np.float32)


def train_epoch(model, dataloader, criterion, optimizer, device, epoch, scaler=None):
    model.train()
    use_amp = scaler is not None
    running_loss = 0.0
    all_preds, all_labels, all_probs = [], [], []

    pbar = tqdm(dataloader, desc=f"Epoch {epoch} [Train]")
    for batch_idx, (images, labels, _) in enumerate(pbar):
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()

        if use_amp:
            with torch.amp.autocast(device_type="cuda"):
                outputs, _ = model(images)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs, _ = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        loss_item = loss.item()
        if np.isnan(loss_item) or np.isinf(loss_item):
            print(f"WARNING: NaN/Inf train loss at batch {batch_idx}")
            continue

        running_loss += loss_item
        if torch.isnan(outputs).any():
            print(f"WARNING: NaN outputs at batch {batch_idx}")
            continue

        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs[:, 1].detach().cpu().numpy())
        pbar.set_postfix({"loss": f"{loss_item:.4f}"})

    avg_loss = running_loss / max(len(dataloader), 1)
    try:
        auc_score = roc_auc_score(all_labels, all_probs) if len(set(all_labels)) > 1 else 0.0
        auc_score = auc_score if not np.isnan(auc_score) else 0.0
    except Exception:
        auc_score = 0.0

    metrics = {
        "accuracy": accuracy_score(all_labels, all_preds),
        "precision": precision_score(all_labels, all_preds, zero_division=0),
        "recall": recall_score(all_labels, all_preds, zero_division=0),
        "f1": f1_score(all_labels, all_preds, zero_division=0),
        "auc": auc_score,
    }
    return avg_loss, metrics


def validate(model, dataloader, criterion, device, epoch):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels, all_probs, all_subject_ids = [], [], [], []

    pbar = tqdm(dataloader, desc=f"Epoch {epoch} [Val]")
    with torch.no_grad():
        for images, labels, subject_ids in pbar:
            images = images.to(device)
            labels = labels.to(device)
            outputs, _ = model(images)
            loss = criterion(outputs, labels)

            loss_item = loss.item()
            if np.isnan(loss_item) or np.isinf(loss_item):
                print("WARNING: NaN/Inf val loss")
                continue
            running_loss += loss_item

            if torch.isnan(outputs).any():
                print("WARNING: NaN val outputs")
                continue

            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].detach().cpu().numpy())
            all_subject_ids.extend(subject_ids)
            pbar.set_postfix({"loss": f"{loss_item:.4f}"})

    avg_loss = running_loss / max(len(dataloader), 1)
    valid_idx = [i for i, p in enumerate(all_probs) if not np.isnan(p)]
    all_probs = [all_probs[i] for i in valid_idx]
    all_labels = [all_labels[i] for i in valid_idx]
    all_preds = [all_preds[i] for i in valid_idx]

    if len(all_probs) == 0 or len(all_labels) == 0:
        return avg_loss, {"accuracy": 0.0, "precision": 0.0, "recall": 0.0, "f1": 0.0, "auc": 0.0}, [], [], []

    try:
        auc_score = roc_auc_score(all_labels, all_probs) if len(set(all_labels)) > 1 else 0.0
        auc_score = auc_score if not np.isnan(auc_score) else 0.0
    except Exception:
        auc_score = 0.0

    metrics = {
        "accuracy": accuracy_score(all_labels, all_preds),
        "precision": precision_score(all_labels, all_preds, zero_division=0),
        "recall": recall_score(all_labels, all_preds, zero_division=0),
        "f1": f1_score(all_labels, all_preds, zero_division=0),
        "auc": auc_score,
    }
    return avg_loss, metrics, all_labels, all_preds, all_probs


def _get_num_slices(args_dict, view):
    """Get total number of slices for a given view based on config."""
    if view == "all":
        return (args_dict.get("num_slices_axial", 80) + 
                args_dict.get("num_slices_coronal", 80) + 
                args_dict.get("num_slices_sagittal", 80))
    elif view == "axial":
        return args_dict.get("num_slices_axial", 80)
    elif view == "coronal":
        return args_dict.get("num_slices_coronal", 80)
    elif view == "sagittal":
        return args_dict.get("num_slices_sagittal", 80)
    return 80


def build_model(model_choice, num_classes, num_slices, embed_dim):
    if model_choice == "densenet":
        return MyDenseNetMultiAttention(num_classes=num_classes, num_slices=num_slices, embed_dim=embed_dim)
    if model_choice == "resnet":
        return MyResNetMultiAttention(num_classes=num_classes, num_slices=num_slices, embed_dim=embed_dim)
    if model_choice == "mobilenet":
        return MyMobileNetMultiAttention(num_classes=num_classes, num_slices=num_slices, embed_dim=embed_dim)
    raise ValueError(f"Unsupported MODEL_CHOICE: {model_choice}")


def plot_fold_curves(history, save_path):
    epochs = np.arange(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    axes[0].plot(epochs, history["train_loss"], label="Train")
    axes[0].plot(epochs, history["val_loss"], label="Val")
    axes[0].set_title("Loss vs Epoch")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].grid(alpha=0.3)
    axes[0].legend()

    axes[1].plot(epochs, history["train_recall"], label="Train")
    axes[1].plot(epochs, history["val_recall"], label="Val")
    axes[1].set_title("Recall vs Epoch")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Recall")
    axes[1].grid(alpha=0.3)
    axes[1].legend()

    axes[2].plot(epochs, history["train_auc"], label="Train")
    axes[2].plot(epochs, history["val_auc"], label="Val")
    axes[2].set_title("AUC vs Epoch")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("AUC")
    axes[2].grid(alpha=0.3)
    axes[2].legend()

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def plot_confusion_matrix_np(labels, preds, class_names, save_path):
    cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(4, 4))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(class_names)))
    ax.set_xticklabels(class_names)
    ax.set_yticks(range(len(class_names)))
    ax.set_yticklabels(class_names)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center")
    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def plot_roc_curve_np(labels, probs, save_path):
    labels = np.asarray(labels)
    probs = np.asarray(probs)
    valid = ~np.isnan(probs)
    labels = labels[valid]
    probs = probs[valid]
    if len(np.unique(labels)) < 2:
        return
    fpr, tpr, _ = roc_curve(labels, probs)
    roc_auc = auc(fpr, tpr)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
    ax.plot([0, 1], [0, 1], "k--")
    ax.set_xlabel("FPR")
    ax.set_ylabel("TPR")
    ax.set_title("ROC Curve")
    ax.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def print_epoch_metrics(epoch, total_epochs, train_loss, val_loss, train_metrics, val_metrics):

    print("[INFO] Train/val utils ready")

        f"Epoch {epoch}/{total_epochs} | "

        f"train_loss={train_loss:.4f} train_acc={train_metrics['accuracy']:.4f} "    )

        f"train_recall={train_metrics['recall']:.4f} train_precision={train_metrics['precision']:.4f} "        f"val_auc={val_metrics['auc']:.4f}"

        f"train_auc={train_metrics['auc']:.4f} | "        f"val_recall={val_metrics['recall']:.4f} val_precision={val_metrics['precision']:.4f} "
        f"val_loss={val_loss:.4f} val_acc={val_metrics['accuracy']:.4f} "

In [ ]:
# ================================================================================
# TRAINING LOOP (CV5 OR SINGLE FOLD)
# ================================================================================

def _get_num_slices(args_dict, view):
    """Get total number of slices for a given view based on config."""
    if view == "all":
        return (
            args_dict.get("num_slices_axial", 80)
            + args_dict.get("num_slices_coronal", 80)
            + args_dict.get("num_slices_sagittal", 80)
        )
    if view == "axial":
        return args_dict.get("num_slices_axial", 80)
    if view == "coronal":
        return args_dict.get("num_slices_coronal", 80)
    if view == "sagittal":
        return args_dict.get("num_slices_sagittal", 80)
    return 80

def train_one_fold(args_dict, fold, model_choice="resnet"):
    set_seed(args_dict["seed"] )
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    num_slices = _get_num_slices(args_dict, args_dict["view"])

    print("\n" + "=" * 70)
    print(f"Training Fold {fold} | View: {args_dict['view']} | Slices: {num_slices} | Model: {model_choice}")
    print("=" * 70)
    print(f"Using device: {device}")

    fold_name = f"fold_{fold}"
    checkpoint_dir = Path(args_dict["checkpoint_dir"]) / fold_name
    log_dir = Path(args_dict["log_dir"]) / fold_name
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    log_dir.mkdir(parents=True, exist_ok=True)

    writer = None
    if HAS_TENSORBOARD and SummaryWriter is not None:
        writer = SummaryWriter(log_dir=str(log_dir))

    use_wandb = bool(args_dict["use_wandb"]) and HAS_WANDB and (wandb is not None)
    if args_dict["use_wandb"] and not use_wandb:
        print("[WARN] wandb requested but unavailable")
    if use_wandb:
        wandb.init(
            project=args_dict["wandb_project"],
            name=f"{args_dict['exp_name']}_fold{fold}",
            config={**args_dict, "model_choice": model_choice, "fold": fold},
            reinit=True,
        )

    train_dataset = OASIS2DDataset(
        data_dir=args_dict["data_dir"],
        fold=fold,
        split="train",
        view=args_dict["view"],
        one_transform=None,
    )
    val_dataset = OASIS2DDataset(
        data_dir=args_dict["data_dir"],
        fold=fold,
        split="val",
        view=args_dict["view"],
        one_transform=None,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=args_dict["batch_size"],
        shuffle=True,
        num_workers=args_dict["num_workers"],
        pin_memory=True,
        persistent_workers=args_dict["num_workers"] > 0,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=args_dict["batch_size"],
        shuffle=False,
        num_workers=args_dict["num_workers"],
        pin_memory=True,
        persistent_workers=args_dict["num_workers"] > 0,
    )

    model = build_model(model_choice, 2, num_slices, args_dict["embed_dim"]).to(device)

    if args_dict["freeze_backbone"]:
        if len(args_dict["trainable_layers"]) == 0:
            for p in model.backbone.parameters():
                p.requires_grad = True
        else:
            for p in model.backbone.parameters():
                p.requires_grad = False
            for name, p in model.backbone.named_parameters():
                if any(layer in name for layer in args_dict["trainable_layers"]):
                    p.requires_grad = True
                    print(f"[unfrozen] {name}")

    if args_dict["use_class_weights"]:
        class_weights = get_class_weights(train_dataset.labels)
        class_weights = torch.FloatTensor(class_weights).to(device)
        criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=args_dict["label_smoothing"])
        print("Class weights:", class_weights.cpu().numpy())
    else:
        criterion = nn.CrossEntropyLoss(label_smoothing=args_dict["label_smoothing"])

    trainable_params = list(filter(lambda p: p.requires_grad, model.parameters()))
    if args_dict["optimizer"] == "adam":
        optimizer = optim.Adam(trainable_params, lr=args_dict["lr"], weight_decay=args_dict["weight_decay"])
    elif args_dict["optimizer"] == "adamw":
        optimizer = optim.AdamW(trainable_params, lr=args_dict["lr"], weight_decay=args_dict["weight_decay"])
    else:
        optimizer = optim.SGD(trainable_params, lr=args_dict["lr"], momentum=0.9, weight_decay=args_dict["weight_decay"])

    if args_dict["scheduler"] == "cosine":
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args_dict["epochs"])
    elif args_dict["scheduler"] == "step":
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=args_dict["step_size"], gamma=args_dict["gamma"])
    elif args_dict["scheduler"] == "plateau":
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.3, patience=10)
    else:
        scheduler = None

    scaler = torch.amp.GradScaler("cuda") if args_dict["use_amp"] and device.type == "cuda" else None

    best_loss = float("inf")
    best_epoch = 0
    best_val_metrics = {"accuracy": 0.0, "precision": 0.0, "recall": 0.0, "f1": 0.0, "auc": 0.0}
    early_stop_counter = 0

    history = {
        "train_loss": [], "val_loss": [],
        "train_acc": [], "val_acc": [],
        "train_precision": [], "val_precision": [],
        "train_recall": [], "val_recall": [],
        "train_auc": [], "val_auc": [],
    }

    last_val_labels, last_val_preds, last_val_probs = [], [], []

    for epoch in range(args_dict["epochs"]):
        train_loss, train_metrics = train_epoch(model, train_loader, criterion, optimizer, device, epoch + 1, scaler)
        val_loss, val_metrics, val_labels, val_preds, val_probs = validate(model, val_loader, criterion, device, epoch + 1)

        if scheduler is not None:
            if args_dict["scheduler"] == "plateau":
                scheduler.step(val_loss)
            else:
                scheduler.step()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_metrics["accuracy"] )
        history["val_acc"].append(val_metrics["accuracy"] )
        history["train_precision"].append(train_metrics["precision"] )
        history["val_precision"].append(val_metrics["precision"] )
        history["train_recall"].append(train_metrics["recall"] )
        history["val_recall"].append(val_metrics["recall"] )
        history["train_auc"].append(train_metrics["auc"] )
        history["val_auc"].append(val_metrics["auc"] )

        # Required per-epoch print: train+val loss/acc/recall/precision/auc
        print_epoch_metrics(epoch + 1, args_dict["epochs"], train_loss, val_loss, train_metrics, val_metrics)

        if writer is not None:
            writer.add_scalar("Loss/train", train_loss, epoch)
            writer.add_scalar("Loss/val", val_loss, epoch)
            writer.add_scalar("Accuracy/train", train_metrics["accuracy"], epoch)
            writer.add_scalar("Accuracy/val", val_metrics["accuracy"], epoch)
            writer.add_scalar("Precision/train", train_metrics["precision"], epoch)
            writer.add_scalar("Precision/val", val_metrics["precision"], epoch)
            writer.add_scalar("Recall/train", train_metrics["recall"], epoch)
            writer.add_scalar("Recall/val", val_metrics["recall"], epoch)
            writer.add_scalar("AUC/train", train_metrics["auc"], epoch)
            writer.add_scalar("AUC/val", val_metrics["auc"], epoch)
            writer.add_scalar("LR", optimizer.param_groups[0]["lr"], epoch)

        if use_wandb:
            wandb.log({
                "epoch": epoch + 1,
                "fold": fold,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "train_acc": train_metrics["accuracy"],
                "val_acc": val_metrics["accuracy"],
                "train_recall": train_metrics["recall"],
                "val_recall": val_metrics["recall"],
                "train_precision": train_metrics["precision"],
                "val_precision": val_metrics["precision"],
                "train_auc": train_metrics["auc"],
                "val_auc": val_metrics["auc"],
                "lr": optimizer.param_groups[0]["lr"],
            })

        # Early stopping by val_loss
        if val_loss < best_loss:
            best_loss = val_loss
            best_epoch = epoch + 1
            best_val_metrics = val_metrics
            early_stop_counter = 0
            save_checkpoint(model, optimizer, epoch + 1, best_loss, checkpoint_dir / "best_model.pth")
            print(f"[SAVE] best_model.pth at epoch={best_epoch}, best_val_loss={best_loss:.4f}")
        else:
            early_stop_counter += 1
            print(f"[EARLY STOP CHECK] {early_stop_counter}/{args_dict['early_stopping_patience']}")
            if args_dict["early_stopping_patience"] > 0 and early_stop_counter >= args_dict["early_stopping_patience"]:
                print("[EARLY STOP] stop by val_loss")
                break

        if (epoch + 1) % args_dict["save_freq"] == 0:
            save_checkpoint(model, optimizer, epoch + 1, best_loss, checkpoint_dir / f"checkpoint_epoch_{epoch + 1}.pth")

        last_val_labels, last_val_preds, last_val_probs = val_labels, val_preds, val_probs

    # Required fold plots: loss, recall, auc (train/val)
    plot_fold_curves(history, log_dir / "fold_curves_loss_recall_auc.png")

    # Keep outputs >= src/2d
    if len(last_val_labels) > 0:
        plot_confusion_matrix_np(last_val_labels, last_val_preds, ["Normal", "Alzheimer"], log_dir / "confusion_matrix.png")
        plot_roc_curve_np(last_val_labels, last_val_probs, log_dir / "roc_curve.png")

    with open(checkpoint_dir / "metrics.json", "w") as f:
        json.dump({
            "fold": fold,
            "best_epoch": int(best_epoch),
            "best_val_loss": float(best_loss),
            "best_val_metrics": best_val_metrics,
            "final_train_loss": float(history["train_loss"][-1]),
            "final_val_loss": float(history["val_loss"][-1]),
            "final_train_acc": float(history["train_acc"][-1]),
            "final_val_acc": float(history["val_acc"][-1]),
        }, f, indent=2)

    if writer is not None:
        writer.close()
    if use_wandb:
        wandb.finish()

    return {
        "fold": fold,
        "best_epoch": best_epoch,
        "best_val_loss": best_loss,
        "best_val_metrics": best_val_metrics,
        "history": history,
        "checkpoint_dir": str(checkpoint_dir),
        "log_dir": str(log_dir),
    }


# ================================================================================
# RUN TRAINING (CV5 OR SINGLE FOLD)
# ================================================================================

set_seed(args["seed"] )

if args["fold"] == -1:
    folds = list(range(NUM_FOLDS))
else:
    folds = [int(args["fold"])]

fold_results = []
for fold in folds:
    result = train_one_fold(args, fold, model_choice=MODEL_CHOICE)
    fold_results.append(result)
    m = result["best_val_metrics"]
    print(f"[FOLD {fold}] best_epoch={result['best_epoch']} best_val_loss={result['best_val_loss']:.4f} best_val_acc={m.get('accuracy', 0.0):.4f} best_val_auc={m.get('auc', 0.0):.4f}")

summary = {
    "model_choice": MODEL_CHOICE,
    "view": args["view"],
    "num_slices_axial": args.get("num_slices_axial", 80),
    "num_slices_coronal": args.get("num_slices_coronal", 80),
    "num_slices_sagittal": args.get("num_slices_sagittal", 80),
    "fold_results": [
        {
            "fold": r["fold"],
            "best_epoch": r["best_epoch"],
            "best_val_loss": float(r["best_val_loss"]),
            "best_val_acc": float(r["best_val_metrics"].get("accuracy", 0.0)),
            "best_val_auc": float(r["best_val_metrics"].get("auc", 0.0)),
        }
        for r in fold_results
    ],
}

if len(fold_results) > 1:
    losses = [r["best_val_loss"] for r in fold_results]
    accs = [r["best_val_metrics"].get("accuracy", 0.0) for r in fold_results]
    aucs = [r["best_val_metrics"].get("auc", 0.0) for r in fold_results]
    summary["cv"] = {
        "mean_best_val_loss": float(np.mean(losses)),
        "std_best_val_loss": float(np.std(losses)),
        "mean_best_val_acc": float(np.mean(accs)),
        "std_best_val_acc": float(np.std(accs)),
        "mean_best_val_auc": float(np.mean(aucs)),
        "std_best_val_auc": float(np.std(aucs)),
    }

summary_path = Path(args["checkpoint_dir"]) / "cv5_summary_notebook.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("\nTRAINING SUMMARY")

In [ ]:
# ================================================================================
# TEST EVALUATION + TEST METRICS
# ================================================================================

best_result = min(fold_results, key=lambda x: x["best_val_loss"])
best_fold = best_result["fold"]
best_ckpt = Path(best_result["checkpoint_dir"]) / "best_model.pth"
print(f"\nEvaluate best fold={best_fold} with checkpoint={best_ckpt}")

if not best_ckpt.exists():
    raise FileNotFoundError(f"Checkpoint missing: {best_ckpt}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_slices = _get_num_slices(args, args["view"])
model = build_model(MODEL_CHOICE, 2, num_slices, args["embed_dim"]).to(device)
ckpt = torch.load(best_ckpt, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

test_dataset = OASIS2DDataset(
    data_dir=args["data_dir"],
    fold=best_fold,
    split="test",
    view=args["view"],
    one_transform=None,
    all_transform=None,
 )
test_loader = DataLoader(
    test_dataset,
    batch_size=args["batch_size"],
    shuffle=False,
    num_workers=args["num_workers"],
    pin_memory=True,
    persistent_workers=args["num_workers"] > 0,
 )

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for images, labels, _ in tqdm(test_loader, desc="Testing"):
        images = images.to(device)
        outputs, _ = model(images)
        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())

test_acc = accuracy_score(all_labels, all_preds)
test_precision = precision_score(all_labels, all_preds, zero_division=0)
test_recall = recall_score(all_labels, all_preds, zero_division=0)
test_f1 = f1_score(all_labels, all_preds, zero_division=0)
try:
    test_auc = roc_auc_score(all_labels, all_probs) if len(set(all_labels)) > 1 else 0.0
except ValueError:
    test_auc = 0.0

print("\nTEST RESULTS")
print(f"Accuracy : {test_acc:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall   : {test_recall:.4f}")
print(f"F1 Score : {test_f1:.4f}")
print(f"AUC      : {test_auc:.4f}")
print("Confusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

test_out_dir = Path(best_result["log_dir"]) / "test_eval"
test_out_dir.mkdir(parents=True, exist_ok=True)
plot_confusion_matrix_np(all_labels, all_preds, ["Normal", "Alzheimer"], test_out_dir / "test_confusion_matrix.png")
plot_roc_curve_np(all_labels, all_probs, test_out_dir / "test_roc_curve.png")

with open(test_out_dir / "test_metrics.json", "w") as f:
    json.dump({
        "best_fold": int(best_fold),
        "model_choice": MODEL_CHOICE,
        "accuracy": float(test_acc),
        "precision": float(test_precision),
        "recall": float(test_recall),
        "f1": float(test_f1),
        "auc": float(test_auc),
    }, f, indent=2)

print("Saved test artifacts to:", test_out_dir)


## Notes
- Notebook is self-contained for Kaggle: only `/kaggle/input/...` and `/kaggle/working/...` are used.
- The config cell includes the full argument set corresponding to `src/2d/train.py`.
- Separate `num_slices_axial`, `num_slices_coronal`, `num_slices_sagittal` control slices per view.
- Every epoch prints: train/val loss, acc, recall, precision, auc.
- Every fold saves curves: train/val loss-epoch, recall-epoch, auc-epoch.
- Early stopping is based on validation loss.
- TensorBoard and Weights & Biases logging are enabled when available.
- Outputs include checkpoints, per-fold metrics, confusion matrix, ROC, CV summary, and test metrics.


## Run Order
1. Run Cell 1: Kaggle config with separate `num_slices_axial`, `num_slices_coronal`, `num_slices_sagittal`.
2. Run Cell 2: imports and optional package checks.
3. Run Cell 3: dataset definitions.
4. Run Cell 4: model definitions.
5. Run Cell 5: train/val utilities, plotting functions, and `_get_num_slices()`.
6. Run Cell 6: training (single fold or CV5).
7. Run Cell 7: test evaluation on best fold.

## Kaggle Outputs
- Checkpoints: `/kaggle/working/checkpoints/2d_cv5_notebook/fold_*/best_model.pth`
- Fold curves: `/kaggle/working/logs/2d_cv5_notebook/fold_*/fold_curves_loss_recall_auc.png`
- Validation plots: `/kaggle/working/logs/2d_cv5_notebook/fold_*/confusion_matrix.png`, `roc_curve.png`
- CV summary: `/kaggle/working/checkpoints/2d_cv5_notebook/cv5_summary_notebook.json`
- Test outputs: `/kaggle/working/logs/2d_cv5_notebook/fold_*/test_eval/`
